# AI Recommendation System — Exploratory Data Analysis

This notebook walks through the sample dataset and demonstrates how the
recommendation engine works under the hood.

**Sections**
1. Load & inspect data
2. Exploratory statistics
3. User-item interaction matrix
4. User-user cosine similarity
5. Item-item cosine similarity
6. Sample recommendations

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/sample_data.csv')
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
df.describe()

## 2. Exploratory Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
df['rating'].hist(bins=10, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Rating Distribution')
axes[0].set_xlabel('Rating')

# Ratings per user
df.groupby('user_id').size().plot(kind='bar', ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title('Ratings per User')
axes[1].set_xlabel('User ID')

# Ratings per item
df.groupby('item_id').size().plot(kind='bar', ax=axes[2], color='mediumseagreen', edgecolor='white')
axes[2].set_title('Ratings per Item')
axes[2].set_xlabel('Item ID')

plt.tight_layout()
plt.show()

## 3. User-Item Interaction Matrix

In [ ]:
matrix = df.pivot_table(index='user_id', columns='item_id', values='rating').fillna(0)
print('Matrix shape:', matrix.shape)
matrix

In [ ]:
plt.figure(figsize=(10, 6))
sns.heatmap(matrix, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.5, cbar_kws={'label': 'Rating'})
plt.title('User-Item Interaction Matrix')
plt.ylabel('User ID')
plt.xlabel('Item ID')
plt.tight_layout()
plt.show()

## 4. User-User Cosine Similarity

In [ ]:
user_sim = cosine_similarity(matrix.values)
user_sim_df = pd.DataFrame(user_sim, index=matrix.index, columns=matrix.index)

plt.figure(figsize=(8, 6))
sns.heatmap(user_sim_df, annot=True, fmt='.2f', cmap='Blues', linewidths=0.5)
plt.title('User-User Cosine Similarity')
plt.tight_layout()
plt.show()

## 5. Item-Item Cosine Similarity

In [ ]:
item_sim = cosine_similarity(matrix.values.T)
item_sim_df = pd.DataFrame(item_sim, index=matrix.columns, columns=matrix.columns)

plt.figure(figsize=(8, 6))
sns.heatmap(item_sim_df, annot=True, fmt='.2f', cmap='Greens', linewidths=0.5)
plt.title('Item-Item Cosine Similarity')
plt.tight_layout()
plt.show()

## 6. Sample Recommendations via the Engine

In [ ]:
from app.recommender import RecommendationEngine

engine = RecommendationEngine()

print('--- Recommendations for user 1 ---')
for r in engine.get_recommendations(1):
    print(f"  item_id={r['item_id']}  predicted_score={r['predicted_score']}")

print()
print('--- Items similar to item 101 ---')
for r in engine.get_similar_items(101):
    print(f"  item_id={r['item_id']}  similarity={r['similarity_score']}")